Lasso Logistic Regression Model

This notebook implements a multi-class logistic regression model with L1 regularization (Lasso) for the classification of:
- Crohn’s Disease (CD)
- Ulcerative Colitis (UC)
- No IBD

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score, classification_report
from sklearn.preprocessing import label_binarize
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
import joblib
from scipy import stats
from sklearn.feature_selection import RFECV
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')
import os
import sys

data_path = "../../data"
os.makedirs(data_path, exist_ok=True)

plots_dir = "../../plots"
os.makedirs(plots_dir, exist_ok=True)

savedmodels_dir = "../../saved_models"
os.makedirs(savedmodels_dir, exist_ok=True)
sys.path.append(os.path.abspath(".."))


model = 'LassologisticRegression'

## Data Availability and Privacy

Due to patient privacy and institutional restrictions, the original electronic medical record (EMR) dataset used in this study cannot be shared. The dataset included in this repository is a synthetic / de-identified sample provided solely to demonstrate the structure of the data

**Note:** Model training and evaluation results presented in the manuscript were obtained using the original dataset and are not derived from the synthetic data included here.


## Data Preparation

Data extraction, cleaning, and structuring were performed prior to model development to create a structured dataset derived from both structured and unstructured EMR data.
- **Structured data** included demographics, diagnosis codes (ICD-10), medications, laboratory values, and healthcare encounters.
- **Unstructured data** were derived from clinical records (e.g., gastroenterology notes, imaging reports, endoscopy, and pathology reports) using keyword-based extraction to capture clinically relevant terms.

In [ ]:
df = pd.read_csv(f"{data_path}/IBD_1200patients.csv")

print("df shape:", df.shape)

In [ ]:
df_raw = df.copy()
df_ids = df_raw[['PatientDurableKey']].copy()

In [ ]:
from utils import drop_cols, LABEL_COL

X = df.drop(columns=drop_cols + [LABEL_COL])
y = df[LABEL_COL]

### Target Encoding

The categorical target variable (CD, UC, No IBD) was transformed into numeric labels using a label encoder.

In [ ]:
# Encode target
le = LabelEncoder()
y = le.fit_transform(y)
for i, cls in enumerate(le.classes_):
    print(f"{cls} → {i}")

## Train–Test Split

The dataset was divided into training and test sets.

- The training set was used for model development, including preprocessing and cross-validation.
- The test set was held out and used only for final model evaluation.

In [ ]:
from utils import random_state, test_size

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

print("Number of samples in X_train:", len(X_train))
print("Number of samples in X_test:", len(X_test))

## Preprocessing
- Categorical variables encoded using OneHotEncoder / OrdinalEncoder
- Continuous variables scaled (for logistic regression)
- All preprocessing transformations were fit on the training dataset and subsequently applied to the evaluation dataset


In [ ]:
from preprocessing import get_logistic_regression_preprocessor

preprocessor =  get_logistic_regression_preprocessor(X_train)

In [ ]:
from sklearn.preprocessing import label_binarize

classes = np.arange(len(le.classes_))
y_binarized = label_binarize(y_test, classes=classes)


In [ ]:
lassopipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(random_state=random_state))])

## Model Training
Hyperparameter tuning via GridSearchCV

In [ ]:
from utils import lassoLRgrid_Params

lassoLR_grid = GridSearchCV(lassopipeline, lassoLRgrid_Params, cv=5, scoring="accuracy", n_jobs=1)

In [ ]:
lassoLR_grid.fit(X_train, y_train)
print("Best parameters:", lassoLR_grid.best_params_)
print("Best CV accuracy: {:.4f}".format(lassoLR_grid.best_score_))

cv_results = lassoLR_grid.cv_results_
best_idx = lassoLR_grid.best_index_
split_scores = np.array([
    cv_results[f"split{i}_test_score"][best_idx]
    for i in range(lassoLR_grid.n_splits_)
])
mean_score = split_scores.mean()
std_err = stats.sem(split_scores)
t_value = stats.t.ppf((1 + 0.95) / 2., len(split_scores) - 1)
ci_lower = mean_score - t_value * std_err
ci_upper = mean_score + t_value * std_err
# print(f"95% CI: ({ci_lower:.4f}, {ci_upper:.4f})")
print(f"Accuracy - 95% CI: {mean_score:.3f} ({ci_lower:.3f}–{ci_upper:.3f})")

best_pipeline = lassoLR_grid.best_estimator_
lassolr_params = {k.replace("model__", ""): v for k, v in lassoLR_grid.best_params_.items()}

## Lasso Logistic Regression with Feature Selection
Feature selection was then applied using SelectFromModel from the trained Lasso model, retaining features with non-zero coefficients.

In [ ]:
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(best_pipeline.named_steps["model"], threshold=1e-6, prefit=True)

In [ ]:
# Preprocess
X_train_transformed = best_pipeline.named_steps["preprocess"].transform(X_train)
X_test_transformed = best_pipeline.named_steps["preprocess"].transform(X_test)

# Apply feature selection
X_train_selected = selector.transform(X_train_transformed)
X_test_selected = selector.transform(X_test_transformed)

print("Selected feature shape:", X_train_selected.shape)

In [ ]:
# Feature names after preprocessing
feature_names = best_pipeline.named_steps['preprocess'].get_feature_names_out()
selected_features = feature_names[selector.get_support()]
print("Number of selected features:", len(selected_features))
clean_features = [f.split("__")[-1] for f in selected_features]

cleanfeatures_df = pd.DataFrame({"Selected Features": clean_features})
cleanfeatures_df.to_csv(f"{plots_dir}/Lassologisticregression_selectedfeatures.csv", index=False)

## Repeated Stratified K-Fold Cross-Validation

To obtain robust and stable performance estimates, we used repeated stratified k-fold cross-validation on the training dataset using the selected features.
Performance metrics, including AUC, sensitivity, specificity, positive predictive value (PPV), and accuracy, are reported as mean values with corresponding 95% confidence intervals.


In [ ]:
from repeated_stratified_kfold import lasso_repeated_stratified_kfold

df_results = lasso_repeated_stratified_kfold(model, selector, X_train, y_train, preprocessor, lassolr_params, le, plots_dir)
df_results

In [ ]:
final_model = LogisticRegression(**lassolr_params, random_state=random_state)

final_model.fit(X_train_selected, y_train)

In [ ]:

# Feature names
feature_names = best_pipeline.named_steps['preprocess'].get_feature_names_out()
selected_features = feature_names[selector.get_support()]
clean_features = [f.split("__")[-1] for f in selected_features]
# Model

coefficients = final_model.coef_
classes = le.classes_

print("Coefficient shape:", coefficients.shape)
print("Number of selected features:", len(selected_features))

# Build results
results = []
for i, class_label in enumerate(classes):
    for j, feature in enumerate(clean_features):
        beta = coefficients[i, j]
        odds_ratio = np.exp(beta)
        results.append({
            "Class": class_label, "Feature": feature, "Coefficient (β)": beta, "Odds Ratio": odds_ratio})
coef_df = pd.DataFrame(results)

# absolute importance for sorting
coef_df["Abs_Coeff"] = coef_df["Coefficient (β)"].abs()
coef_df = coef_df.sort_values(
    by=["Class", "Abs_Coeff"],
    ascending=[True, False])

os.makedirs(plots_dir, exist_ok=True)
file_path = f"{plots_dir}/lassologisticregression_coefficients.csv"
coef_df.to_csv(file_path, index=False)

In [ ]:

y_pred = final_model.predict(X_test_selected)
y_prob = final_model.predict_proba(X_test_selected)

In [ ]:
import joblib
import os


joblib.dump(best_pipeline.named_steps['preprocess'], f"{savedmodels_dir}/lassologisticregression_preprocessor.pkl")
joblib.dump(selector, f"{savedmodels_dir}/lassologisticregression_selector.pkl")
joblib.dump(final_model, f"{savedmodels_dir}/lassologisticregression_model.pkl")


In [ ]:
y_pred_original = le.inverse_transform(y_pred)
y_test_original = le.inverse_transform(y_test)

patient_ids = df_ids.loc[X_test.index, 'PatientDurableKey']

assert len(patient_ids) == len(y_pred_original)
results_df = pd.DataFrame({
    'patient_id': patient_ids.values,
    'Actual IBD': y_test_original,
    'Predicted IBD': y_pred_original
})

results_df.to_csv(f"{plots_dir}/Lassologisticregression_predictionresults.csv", index=False)

## Model Evaluation on Test Data
Model performance was evaluated on an independent test dataset

The following metrics were used:
- ROC-AUC
- Confusion matrix
- Sensitivity and specificity

In [ ]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
from evaluation_plots import display_confusion_matrix, precision_per_class, sensitivity_specificity_per_class,sensitivity_specificity_cduc_noibd, display_roc_curves
cm = display_confusion_matrix(model, y_test, y_pred, le, plots_dir)

In [ ]:
precision_per_class(cm, le)

In [ ]:
sensitivity_specificity_per_class(model,cm,le, plots_dir)

In [ ]:
sensitivity_specificity_cduc_noibd(cm, le)

In [ ]:
auc_macro = roc_auc_score(y_binarized, y_prob, average="macro", multi_class="ovr")
auc_weighted = roc_auc_score(y_binarized, y_prob, average="weighted", multi_class="ovr")

print(f"\nMacro AUC: {auc_macro:.3f}")
print(f"Weighted AUC: {auc_weighted:.3f}")

In [ ]:
display_roc_curves(model, y_test, y_prob, plots_dir, le)